[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/92-agent-sandboxing-e2b/e2b_sandboxing_workbook.ipynb)

# 92 · Agent Sandboxing — Safe Code Execution with E2B

## Isolated MicroVM Code Execution for AI Agents

**⏱ ~90 min**

When an AI agent generates and executes code, running that code in the host process with `exec()` is dangerous — the agent can overwrite files, leak secrets, or run arbitrary system commands. E2B (e2b.dev) provides isolated microVM sandboxes: each code execution runs in a fresh container that has no access to your host filesystem, environment variables, or network unless you explicitly grant it. This workbook builds the **generate → sandbox → interpret** LangGraph pipeline.

---

### Workshop Roadmap

| Part | Topic | Concept |
|------|-------|---------|
| 1 | Why sandboxing matters | `exec()` risks vs. isolated microVM |
| 2 | E2B API basics | `Sandbox()`, `run_code()`, logs, errors |
| 3 | The `generate_code` node | Separation of generation and execution |
| 4 | The `execute_in_sandbox` node | Structured stdout/stderr/error capture |
| 5 | Full pipeline | generate → sandbox → interpret |
| 6 | Exercises | Buggy code, file-write isolation |

> **Note:** This workbook requires both `OPENAI_API_KEY` and `E2B_API_KEY`. Get a free E2B key at https://e2b.dev/dashboard

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "python-dotenv", "e2b-code-interpreter"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    os.environ["E2B_API_KEY"] = userdata.get("E2B_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

openai_ok = bool(os.environ.get("OPENAI_API_KEY", "")) and os.environ.get("OPENAI_API_KEY", "").startswith("sk-")
e2b_ok = bool(os.environ.get("E2B_API_KEY", ""))
print(f"OPENAI_API_KEY: {openai_ok}")
print(f"E2B_API_KEY:    {e2b_ok}")
if not openai_ok:
    print("  ACTION: set OPENAI_API_KEY (sk-...)")
if not e2b_ok:
    print("  ACTION: set E2B_API_KEY — get one free at https://e2b.dev/dashboard")

---

## Part 1 — Why Sandboxing Matters

### `exec()` risks vs. isolated microVM

`exec()` in Python gives the executed code access to the interpreter's full environment — it can read any in-memory variable, write to disk, call `subprocess`, import network libraries. In an agentic context this is catastrophic: LLM-generated code could read your API keys from memory, exfiltrate data, or destroy files.

**E2B's microVM approach:** each execution gets a fresh container with its own filesystem and network namespace. The container starts clean, runs your code, and is destroyed after — taking everything with it.

| Capability | Local `exec()` | E2B Sandbox |
|-----------|---------------|-------------|
| Host filesystem access | ✅ Full read/write | ❌ Isolated container |
| Host env var access | ✅ Via `os.environ` | ❌ Clean environment |
| Host network access | ✅ Unrestricted | ✅ Configurable (off by default) |
| Cleanup after run | ❌ Manual | ✅ Container destroyed automatically |
| Cost | Free (but risky) | ~$0.000X per execution |
| Latency | ~0ms | ~1-3s (cold start) |

In [ ]:
# Demonstrate exec() visibility into host environment
# This shows WHY sandboxing matters — exec() can see everything

sensitive_variable = "super_secret_api_key_12345"  # simulates an env var in memory

print("=== exec() can see ALL host Python variables ===\n")
exec("print('Variables visible to exec():', [k for k in dir() if not k.startswith('_')][:10])")
exec("print('Can see sensitive_variable:', sensitive_variable)")

print()
print("=== E2B sandbox isolation ===")
print("In E2B: the code runs in a separate process/container")
print("  - Cannot read host Python variables")
print("  - Cannot read host environment variables")
print("  - Cannot write to host filesystem")
print("  - Container is destroyed after execution")
print("  - Each sandbox.run_code() call is independent")

---

## Part 2 — E2B API Basics

### `Sandbox()`, `run_code()`, `execution.logs`, `execution.error`

The E2B Python SDK is straightforward:

```python
from e2b_code_interpreter import Sandbox

with Sandbox() as sb:
    result = sb.run_code("print('hello')")
    print(result.logs.stdout)  # ['hello\n']
    print(result.error)        # None on success
```

**The result object:**
- `result.logs.stdout` — list of strings, one per print statement
- `result.logs.stderr` — list of strings for stderr output
- `result.error` — `None` on success, or an error object with `.name`, `.value`, `.traceback`
- `result.results` — list of rich outputs (plots, DataFrames, etc.)

The `with Sandbox() as sb:` context manager guarantees the container is destroyed when the block exits — even if an exception occurs.

In [ ]:
from e2b_code_interpreter import Sandbox

print("Creating E2B sandbox...")
with Sandbox() as sb:
    # Simple stdout test
    result = sb.run_code("print('Hello from sandbox!')\nprint(2 + 2)")
    print("stdout:", result.logs.stdout)
    print("stderr:", result.logs.stderr)
    print("error:", result.error)
    print()

    # Math computation
    result2 = sb.run_code("import math\nprint(math.pi)\nprint(math.sqrt(144))")
    print("Math stdout:", result2.logs.stdout)

    # Intentional error
    result3 = sb.run_code("1 / 0")
    print()
    print("Error test:")
    print("  error type:", result3.error.name if result3.error else None)
    print("  error value:", result3.error.value if result3.error else None)
    print("  stdout:", result3.logs.stdout)

print("\nSandbox closed — container destroyed.")

---

## Part 3 — The `generate_code` Node

### Separation of generation and execution

**Never generate and execute code in the same step.** The separation matters:

- The **generate** step is cheap to retry (just re-call the LLM)
- The **execute** step is isolated (runs in the sandbox)
- Separation lets you **log generated code** before running it — useful for auditing
- You can add a **human-in-the-loop approval** step between generate and execute

The generate node calls `gpt-4o-mini` with a prompt that instructs the model to return only executable Python code. We strip markdown code fences if present, since LLMs sometimes wrap output in ` ```python ` blocks.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import re

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # deterministic code generation

CODEGEN_PROMPT = '''Write Python code to solve this task.
Return ONLY the Python code — no markdown, no explanations, just the code.
The code must print its results using print() statements.

Task: {task}'''

def extract_code(raw: str) -> str:
    """Strip markdown code fences if present."""
    # Remove ```python ... ``` or ``` ... ``` wrappers
    match = re.search(r'```(?:python)?\s*(.*?)```', raw, re.DOTALL)
    if match:
        return match.group(1).strip()
    return raw.strip()

def generate_code(task: str) -> str:
    """Generate Python code for a task (not yet executed)."""
    resp = llm.invoke([HumanMessage(content=CODEGEN_PROMPT.format(task=task))])
    code = extract_code(resp.content)
    return code

sample_tasks = [
    "Calculate the first 10 Fibonacci numbers",
    "Find all prime numbers up to 50",
    "Sort this list and find the median: [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]"
]

for task in sample_tasks:
    code = generate_code(task)
    print(f"Task: {task}")
    print("Generated code:")
    for line in code.split('\n'):
        print(f"  {line}")
    print()

---

## Part 4 — The `execute_in_sandbox` Node

### Structured stdout/stderr/error capture

The sandbox node runs code and returns a **structured result** with:
- `stdout` — all print() output joined into a single string
- `stderr` — warnings and debug output
- `error` — `None` on success, or `"ErrorType: message"` on failure
- `success` — boolean flag for routing decisions

**Always capture all three** — never assume code succeeds. The `success` field lets the LangGraph router decide whether to retry, fix the code, or proceed to interpretation.

We use a `TypedDict` for the return type so downstream nodes know exactly what fields to expect.

In [ ]:
from e2b_code_interpreter import Sandbox
from typing_extensions import TypedDict

class ExecutionResult(TypedDict):
    stdout: str
    stderr: str
    error: str | None
    success: bool

def execute_in_sandbox(code: str) -> ExecutionResult:
    """Execute code in an E2B sandbox and return structured results."""
    with Sandbox() as sb:
        result = sb.run_code(code)
        stdout = "\n".join(result.logs.stdout) if result.logs.stdout else ""
        stderr = "\n".join(result.logs.stderr) if result.logs.stderr else ""
        if result.error:
            error_msg = f"{result.error.name}: {result.error.value}"
        else:
            error_msg = None
        return {
            "stdout": stdout,
            "stderr": stderr,
            "error": error_msg,
            "success": result.error is None
        }

# Test with good code and bad code
test_cases = [
    ("Good code", "nums = list(range(1, 11))\nprint('Sum:', sum(nums))\nprint('Product:', __import__('math').prod(nums))"),
    ("Division error", "x = 10\ny = 0\nprint(x / y)"),
    ("Import error", "import nonexistent_package\nprint('hi')")
]

for label, code in test_cases:
    print(f"=== {label} ===")
    print("Code:", code[:80])
    result = execute_in_sandbox(code)
    print(f"  success: {result['success']}")
    if result["stdout"]:
        print(f"  stdout: {result['stdout'][:100]}")
    if result["error"]:
        print(f"  error: {result['error']}")
    print()

---

## Part 5 — Full Pipeline: Generate → Sandbox → Interpret

### Wiring the three nodes into a LangGraph workflow

The complete pipeline has three nodes:

1. **`generate`** — calls the LLM to produce Python code for the task
2. **`sandbox`** — executes the code in an E2B microVM, captures results
3. **`interpret`** — calls the LLM again to explain the output in plain English

The `SandboxState` TypedDict threads all data through the graph. Each node receives the full state and returns a dict of fields to update.

**Key insight:** The interpret node sees both the generated code AND the execution output. If the code errored, the LLM can still explain what went wrong — making the pipeline robust to failures.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict

class SandboxState(TypedDict):
    task: str
    generated_code: str
    stdout: str
    stderr: str
    error: str | None
    interpretation: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

INTERPRET_PROMPT = '''A user asked: {task}

The following Python code was generated and executed in a sandbox:
```python
{code}
```

Execution output:
stdout: {stdout}
error: {error}

Summarize what the code computed and answer the user's task in plain English.'''

def generate_code_node(state: SandboxState) -> dict:
    code = generate_code(state["task"])
    print(f"Generated {len(code.split(chr(10)))} lines of code")
    return {"generated_code": code}

def sandbox_node(state: SandboxState) -> dict:
    print("Executing in E2B sandbox...")
    result = execute_in_sandbox(state["generated_code"])
    print(f"  success={result['success']}, stdout_len={len(result['stdout'])}")
    return {
        "stdout": result["stdout"],
        "stderr": result["stderr"],
        "error": result["error"]
    }

def interpret_node(state: SandboxState) -> dict:
    prompt = INTERPRET_PROMPT.format(
        task=state["task"],
        code=state["generated_code"],
        stdout=state["stdout"] or "(empty)",
        error=state["error"] or "(none)"
    )
    resp = llm.invoke([HumanMessage(content=prompt)])
    return {"interpretation": resp.content.strip()}

g = StateGraph(SandboxState)
g.add_node("generate", generate_code_node)
g.add_node("sandbox", sandbox_node)
g.add_node("interpret", interpret_node)
g.set_entry_point("generate")
g.add_edge("generate", "sandbox")
g.add_edge("sandbox", "interpret")
g.add_edge("interpret", END)
app = g.compile()

tasks = [
    "Calculate the first 15 Fibonacci numbers and find which ones are also prime",
    "Find all perfect numbers less than 1000"
]

for task in tasks:
    print(f"\nTask: {task}")
    print("=" * 60)
    result = app.invoke({
        "task": task, "generated_code": "", "stdout": "",
        "stderr": "", "error": None, "interpretation": ""
    })
    print(f"\nInterpretation:\n{result['interpretation']}")

---

## Part 6 — Exercises

### Exercise (a): Buggy code through the pipeline

Send intentionally broken code through the full pipeline and observe how the error is captured and interpreted.

**Goal:** understand what `result["error"]` contains, and how the interpret node handles failures.

### Exercise (b): File-write isolation test

Ask the agent to write a file to `/tmp/sandbox_test.txt`, then check whether that file appears on your host machine.

**Goal:** confirm that E2B's container filesystem is completely isolated from your host.

In [ ]:
# Exercise (a): send buggy code through the pipeline
buggy_task = "Calculate factorial of 'hello'"  # will generate code that errors

# TODO: run app.invoke({task: buggy_task, ...}) and inspect result["error"]
# What does the interpretation say when code fails?

# Exercise (b): file write isolation test
import os
file_write_task = "Write the string 'sandbox_was_here' to a file called /tmp/sandbox_test.txt"

# TODO: run app.invoke({task: file_write_task, ...})
# Then check: os.path.exists("/tmp/sandbox_test.txt") — should be False
# The file exists INSIDE the sandbox container, not on your host

print("Exercise (a): buggy_task defined — run through pipeline and inspect error")
print("Exercise (b): file_write_task defined — verify no host file is created")

---

## Answer Key

### Exercise (a)

When the LLM generates `math.factorial('hello')`, Python raises a `TypeError` inside the sandbox. The `execute_in_sandbox` function captures this as:

```
error: TypeError: 'str' object cannot be interpreted as an integer
```

The `interpret_node` receives this error in its prompt and explains to the user: *"The code failed because `factorial()` requires an integer argument, not a string like 'hello'. Factorial is only defined for non-negative integers."*

Key takeaway: the pipeline degrades gracefully — errors are surfaced to the user in plain language rather than crashing silently.

### Exercise (b)

The generated code runs `open('/tmp/sandbox_test.txt', 'w').write('sandbox_was_here')` successfully **inside the E2B container**. The stdout confirms the write succeeded.

But on your host:
```python
os.path.exists("/tmp/sandbox_test.txt")  # → False
```

The file was created inside the microVM's filesystem. When the `with Sandbox()` block closes, the entire container — including its filesystem — is destroyed. Nothing persists to your host machine.

This is the **core safety guarantee** of E2B: LLM-generated code can read/write/delete files freely inside the sandbox, and none of it touches your host. This is what makes it safe to execute untrusted agent-generated code in production.